# 11. Upload dos adaptadores para o Hugging Face Hub

Este notebook prepara e executa o upload dos adaptadores LoRA/QLoRA treinados no experimento para um repositório do Hugging Face Hub.

A ideia principal é publicar **os adaptadores**, não o modelo completo. Isso é importante porque o modelo base usado no experimento é:

```text
meta-llama/Llama-3.1-8B-Instruct
```

Esse modelo base possui acesso controlado no Hugging Face. Portanto, este notebook não deve redistribuir os pesos completos do modelo base. Em vez disso, ele publica apenas os arquivos produzidos pelo fine-tuning dos cenários C2, C3 e C4.

Os adaptadores esperados são:

```text
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Cada cenário possui três réplicas experimentais:

```text
seed_42
seed_123
seed_2026
```

O resultado recomendado é um único repositório de modelo no Hugging Face contendo a estrutura:

```text
c2_struq_sft/
  seed_42/
  seed_123/
  seed_2026/

c3_secalign_dpo/
  seed_42/
  seed_123/
  seed_2026/

c4_ih_sft/
  seed_42/
  seed_123/
  seed_2026/

README.md
experiment_adapters_manifest.json
```

Por segurança, o notebook começa em modo `DRY_RUN=True`. Nesse modo, ele valida os arquivos e mostra o que seria enviado, mas não cria repositório nem faz upload. Para executar o upload de fato, é necessário revisar as configurações e alterar explicitamente para:

```python
DRY_RUN = False
```

## 0. Política de espaço: upload sem cópia intermediária

Este notebook foi planejado para ambientes com espaço em disco limitado. Por isso, ele **não cria uma pasta temporária com cópia dos adaptadores** antes do upload.

A lógica usada é:

```text
1. validar os diretórios originais dos adaptadores;
2. gerar apenas metadados leves localmente, como README, manifesto e exemplos de carregamento;
3. enviar os adaptadores diretamente dos caminhos originais usando upload_folder;
4. não duplicar adapter_model.safetensors, checkpoints ou outros arquivos pesados em exports/.
```

Os metadados leves gerados por este notebook ficam em:

```text
exports/huggingface_upload/
```

Esses arquivos são pequenos e servem para auditoria do upload. Já os pesos dos adaptadores continuam somente nos diretórios originais, por exemplo:

```text
adapters/struq/seed_42/
adapters/secalign/seed_42/
adapters/ih/seed_42/
```

Durante o upload, o Hugging Face Hub lê diretamente esses diretórios originais. Assim, o notebook evita duplicar arquivos grandes e reduz o risco de falta de espaço em disco.


## 1. Por que subir adaptadores, e não o modelo completo?

Os cenários treinados neste experimento usam LoRA/QLoRA. Esse tipo de treinamento não cria uma cópia completa do modelo base; ele cria um conjunto menor de pesos adicionais, chamados adaptadores.

Subir apenas os adaptadores tem várias vantagens:

```text
- reduz muito o tamanho do upload;
- evita duplicar os pesos do modelo base;
- facilita rastrear qual parte é o modelo original e qual parte é a adaptação experimental;
- mantém a compatibilidade com o ecossistema PEFT;
- evita redistribuir diretamente pesos de um modelo gated.
```

Quem quiser usar um adaptador publicado precisará carregar primeiro o mesmo modelo base e depois aplicar o adaptador correspondente.

A lógica de carregamento posterior será parecida com:

```python
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = PeftModel.from_pretrained(
    base_model,
    "usuario/pi-defense-adapters",
    subfolder="c2_struq_sft/seed_42",
)
```

Assim, o repositório no Hugging Face funciona como um pacote de adaptadores experimentais, e não como um modelo completo independente.

## 2. Imports, caminhos e utilitários

Nesta etapa, são definidos os imports, diretórios e funções auxiliares usados pelo notebook.

Os arquivos gerados por esta etapa ficam em três lugares:

```text
exports/huggingface_upload/
logs/huggingface_upload/
manifests/huggingface_upload/
```

Esses diretórios armazenam apenas metadados leves, como manifesto, README temporário, logs de upload e índice dos adaptadores. Os pesos dos adaptadores continuam no diretório original `adapters/` e são enviados diretamente para o Hugging Face.

In [1]:
from __future__ import annotations

import json
import math
import time
import traceback
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd

from huggingface_hub import HfApi, login, whoami

In [2]:
PROJECT_ROOT = Path("/workspace/pi-defense-exp")

# Este diretório guarda apenas arquivos leves gerados pelo notebook 11.
# Ele NÃO recebe cópias dos adaptadores ou pesos treinados.
UPLOAD_METADATA_DIR = PROJECT_ROOT / "exports" / "huggingface_upload"
EXPORT_DIR = UPLOAD_METADATA_DIR  # alias mantido para compatibilidade com as células abaixo

LOG_DIR = PROJECT_ROOT / "logs" / "huggingface_upload"
MANIFEST_DIR = PROJECT_ROOT / "manifests" / "huggingface_upload"

for directory in [UPLOAD_METADATA_DIR, LOG_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EVENTS_LOG_PATH = LOG_DIR / "11_upload_adapters_to_huggingface_events.jsonl"

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
EXPERIMENT_SEEDS = [42, 123, 2026]

# Política de espaço: não criar staging com cópia dos adaptadores.
NO_INTERMEDIATE_ADAPTER_COPY = True

print("Project root:", PROJECT_ROOT)
print("Upload metadata dir:", UPLOAD_METADATA_DIR)
print("Log dir:", LOG_DIR)
print("Manifest dir:", MANIFEST_DIR)
print("Base model:", BASE_MODEL_ID)
print("Experiment seeds:", EXPERIMENT_SEEDS)
print("No intermediate adapter copy:", NO_INTERMEDIATE_ADAPTER_COPY)

Project root: /workspace/pi-defense-exp
Upload metadata dir: /workspace/pi-defense-exp/exports/huggingface_upload
Log dir: /workspace/pi-defense-exp/logs/huggingface_upload
Manifest dir: /workspace/pi-defense-exp/manifests/huggingface_upload
Base model: meta-llama/Llama-3.1-8B-Instruct
Experiment seeds: [42, 123, 2026]
No intermediate adapter copy: True


In [3]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value):
    if isinstance(value, dict):
        return {str(key): sanitize_json_value(val) for key, val in value.items()}

    if isinstance(value, list):
        return [sanitize_json_value(item) for item in value]

    if isinstance(value, tuple):
        return [sanitize_json_value(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"

    return value


def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            sanitize_json_value(data),
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(
                sanitize_json_value(row),
                ensure_ascii=False,
                allow_nan=False,
            )
            + "\n"
        )


def log_event(event_type: str, payload: dict | None = None) -> None:
    row = {
        "timestamp_utc": utc_now(),
        "event_type": event_type,
    }

    if payload:
        row.update(payload)

    append_jsonl(EVENTS_LOG_PATH, row)


def get_directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0

    total = 0

    for item in path.rglob("*"):
        if item.is_file():
            total += item.stat().st_size

    return total


def count_files(path: Path) -> int:
    if not path.exists():
        return 0

    return sum(1 for item in path.rglob("*") if item.is_file())


def bytes_to_mb(size_bytes: int) -> float:
    return size_bytes / (1024 * 1024)

## 3. Configuração do upload

Esta seção define o repositório de destino no Hugging Face.

Antes de executar upload real, altere:

```python
HF_REPO_ID = "seu-usuario/pi-defense-adapters"
```

ou use uma organização:

```python
HF_REPO_ID = "sua-organizacao/pi-defense-adapters"
```

O notebook começa com:

```python
DRY_RUN = True
```

Enquanto esse valor estiver ativo, o notebook apenas valida e simula o upload. Para enviar os adaptadores de fato, revise as configurações e altere para:

```python
DRY_RUN = False
```

A recomendação inicial é criar o repositório como privado. Depois de revisar README, model card, arquivos enviados e eventuais restrições de licença, ele pode ser tornado público manualmente no Hub.

In [4]:
HF_REPO_ID = "leinha/pi-defense-adapters"
HF_REPO_PRIVATE = True

DRY_RUN = False

UPLOAD_README = True
UPLOAD_ADAPTER_MANIFEST = True
UPLOAD_ADAPTERS = True
VERIFY_REMOTE_FILES = True

# Por padrão, não enviamos checkpoints intermediários para economizar espaço.
UPLOAD_IGNORE_PATTERNS = [
    "checkpoint-*",
    "**/checkpoint-*",
    "runs/**",
    "**/runs/**",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
    "training_args.bin",
    "*.log",
    "__pycache__/**",
    ".ipynb_checkpoints/**",
]

print("HF_REPO_ID:", HF_REPO_ID)
print("HF_REPO_PRIVATE:", HF_REPO_PRIVATE)
print("DRY_RUN:", DRY_RUN)
print("UPLOAD_IGNORE_PATTERNS:", UPLOAD_IGNORE_PATTERNS)

HF_REPO_ID: leinha/pi-defense-adapters
HF_REPO_PRIVATE: True
DRY_RUN: False
UPLOAD_IGNORE_PATTERNS: ['checkpoint-*', '**/checkpoint-*', 'runs/**', '**/runs/**', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'training_args.bin', '*.log', '__pycache__/**', '.ipynb_checkpoints/**']


## 4. Autenticação no Hugging Face

Para criar repositórios ou fazer upload, o ambiente precisa estar autenticado no Hugging Face.

O notebook tenta primeiro detectar uma autenticação já existente. Se ela não existir, solicita um token com `getpass`, para evitar que o token apareça no notebook.

O token deve ter permissão de escrita no Hub. Se o repositório for criado dentro de uma organização, a conta autenticada também precisa ter permissão para criar ou escrever nesse namespace.

O token não é salvo nos manifestos, logs ou README.

In [5]:
api = HfApi()

try:
    hf_user = whoami()
    HF_USERNAME = hf_user.get("name")
    print("Hugging Face login detectado.")
    print("User:", HF_USERNAME)
except Exception:
    print("Login Hugging Face não detectado.")
    hf_token = getpass("Cole seu token do Hugging Face: ")
    
    login(
        token=hf_token,
        add_to_git_credential=False,
    )
    
    hf_user = whoami()
    HF_USERNAME = hf_user.get("name")
    print("Login realizado para o ambiente atual.")
    print("User:", HF_USERNAME)

log_event("huggingface_auth_checked", {"user": HF_USERNAME})

Hugging Face login detectado.
User: leinha


## 5. Plano de adaptadores a enviar

Esta etapa define o mapeamento entre os IDs operacionais do experimento e os diretórios locais dos adaptadores.

O notebook espera encontrar os adaptadores em:

```text
adapters/struq/seed_<seed>/
adapters/secalign/seed_<seed>/
adapters/ih/seed_<seed>/
```

E enviá-los para o repositório Hugging Face como:

```text
c2_struq_sft/seed_<seed>/
c3_secalign_dpo/seed_<seed>/
c4_ih_sft/seed_<seed>/
```

Essa separação preserva a distinção entre o nome local usado durante o treinamento e o ID metodológico usado na documentação do experimento.

In [6]:
SCENARIO_ADAPTER_PLAN = {
    "c2_struq_sft": {
        "label": "C2 — StruQ-like SFT",
        "local_root": PROJECT_ROOT / "adapters" / "struq",
        "repo_root": "c2_struq_sft",
        "method": "sft",
    },
    "c3_secalign_dpo": {
        "label": "C3 — SecAlign-like DPO",
        "local_root": PROJECT_ROOT / "adapters" / "secalign",
        "repo_root": "c3_secalign_dpo",
        "method": "dpo",
    },
    "c4_ih_sft": {
        "label": "C4 — Instruction-Hierarchy-like SFT",
        "local_root": PROJECT_ROOT / "adapters" / "ih",
        "repo_root": "c4_ih_sft",
        "method": "sft",
    },
}

adapter_records = []

for scenario_id, info in SCENARIO_ADAPTER_PLAN.items():
    for seed in EXPERIMENT_SEEDS:
        local_path = info["local_root"] / f"seed_{seed}"
        path_in_repo = f"{info['repo_root']}/seed_{seed}"

        size_bytes = get_directory_size_bytes(local_path)

        adapter_records.append({
            "scenario_id": scenario_id,
            "scenario_label": info["label"],
            "method": info["method"],
            "seed": seed,
            "local_path": local_path,
            "path_in_repo": path_in_repo,
            "exists": local_path.exists(),
            "file_count": count_files(local_path),
            "size_bytes": size_bytes,
            "size_mb": bytes_to_mb(size_bytes),
            "has_adapter_config": (local_path / "adapter_config.json").exists(),
            "has_adapter_safetensors": (local_path / "adapter_model.safetensors").exists(),
            "has_adapter_bin": (local_path / "adapter_model.bin").exists(),
        })

adapter_df = pd.DataFrame(adapter_records)
display(adapter_df)

,scenario_id,scenario_label,method,seed,local_path,path_in_repo,exists,file_count,size_bytes,size_mb,has_adapter_config,has_adapter_safetensors,has_adapter_bin
0,c2_struq_sft,C2 — StruQ-like SFT,sft,42,/workspace/pi-defense-exp/adapters/struq/seed_42,c2_struq_sft/seed_42,True,29,639878716,610.235897,True,True,False
1,c2_struq_sft,C2 — StruQ-like SFT,sft,123,/workspace/pi-defense-exp/adapters/struq/seed_123,c2_struq_sft/seed_123,True,29,639878860,610.236034,True,True,False
2,c2_struq_sft,C2 — StruQ-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/struq/seed_...,c2_struq_sft/seed_2026,True,29,639878734,610.235914,True,True,False
3,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,42,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_42,True,29,639881927,610.238959,True,True,False
4,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,123,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_123,True,29,639881948,610.238979,True,True,False
5,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,2026,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_2026,True,29,639881876,610.238911,True,True,False
6,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,42,/workspace/pi-defense-exp/adapters/ih/seed_42,c4_ih_sft/seed_42,True,29,639878726,610.235907,True,True,False
7,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,123,/workspace/pi-defense-exp/adapters/ih/seed_123,c4_ih_sft/seed_123,True,29,639878843,610.236018,True,True,False
8,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/ih/seed_2026,c4_ih_sft/seed_2026,True,29,639878717,610.235898,True,True,False


In [7]:
# Garantia de política de espaço:
# nenhum adaptador será copiado para UPLOAD_METADATA_DIR.
# O upload usa os diretórios originais informados em SCENARIO_ADAPTER_PLAN.

for scenario_id, info in SCENARIO_ADAPTER_PLAN.items():
    adapter_root = Path(info["local_root"]).resolve()
    metadata_root = UPLOAD_METADATA_DIR.resolve()

    if str(adapter_root).startswith(str(metadata_root)):
        raise RuntimeError(
            "adapter_root está dentro de UPLOAD_METADATA_DIR. "
            "Isso indicaria cópia/staging de arquivos pesados, o que deve ser evitado. "
            f"scenario_id={scenario_id}, adapter_root={adapter_root}"
        )

print("Política no-copy validada: os adaptadores serão lidos dos diretórios originais.")


Política no-copy validada: os adaptadores serão lidos dos diretórios originais.


## 6. Validação dos adaptadores

Antes de criar o repositório ou enviar arquivos, o notebook verifica se todos os adaptadores esperados existem.

Para cada combinação de cenário e seed, são esperados pelo menos:

```text
adapter_config.json
adapter_model.safetensors
```

ou, em alguns casos:

```text
adapter_config.json
adapter_model.bin
```

Se algum adaptador estiver ausente, o notebook interrompe a execução. Isso evita publicar um repositório incompleto sem perceber.

In [8]:
missing_adapters_df = adapter_df.loc[~adapter_df["exists"]].copy()

if not missing_adapters_df.empty:
    display(missing_adapters_df)
    raise FileNotFoundError(
        "Alguns diretórios de adaptadores não foram encontrados. "
        "Verifique se o notebook 04 foi executado com sucesso."
    )

missing_required_files_df = adapter_df.loc[
    ~adapter_df["has_adapter_config"]
    | (~adapter_df["has_adapter_safetensors"] & ~adapter_df["has_adapter_bin"])
].copy()

if not missing_required_files_df.empty:
    display(missing_required_files_df)
    raise FileNotFoundError(
        "Alguns adaptadores não possuem adapter_config.json e/ou arquivo de pesos do adaptador."
    )

total_adapter_size_mb = adapter_df["size_mb"].sum()

print("Todos os adaptadores esperados foram encontrados.")
print(f"Total estimado dos diretórios de adaptadores: {total_adapter_size_mb:.2f} MB")

log_event("adapter_validation_completed", {
    "adapter_count": len(adapter_records),
    "total_adapter_size_mb": total_adapter_size_mb,
})

Todos os adaptadores esperados foram encontrados.
Total estimado dos diretórios de adaptadores: 5492.13 MB


## 7. Gerar README/model card do repositório

O repositório no Hugging Face precisa de uma documentação própria. Esta célula gera um `README.md` com:

```text
- modelo base;
- cenários publicados;
- seeds disponíveis;
- estrutura do repositório;
- exemplo de carregamento com PEFT;
- limitações;
- observação sobre uso experimental.
```

Esse README será enviado para a raiz do repositório no Hugging Face.

Antes de tornar o repositório público, revise o README manualmente para confirmar se o texto, o nome do repositório e as restrições de uso estão adequados.

In [9]:
readme_path = EXPORT_DIR / "README.md"

model_card = f"""---
base_model: {BASE_MODEL_ID}
library_name: peft
tags:
- peft
- lora
- qlora
- prompt-injection
- safety
- instruction-following
- experimental
---

# Prompt-Injection Defense Adapters

This repository contains LoRA/QLoRA adapters trained for an experimental evaluation of prompt-injection defenses.

The base model is:

```text
{BASE_MODEL_ID}
```

The base model is not included in this repository. Users must have access to the base model in order to load these adapters.

## Available adapters

The repository contains adapters for three trained scenarios:

| Scenario | Method | Seeds |
|---|---|---|
| C2 — StruQ-like SFT | Supervised fine-tuning | 42, 123, 2026 |
| C3 — SecAlign-like DPO | Preference optimization | 42, 123, 2026 |
| C4 — Instruction-Hierarchy-like SFT | Supervised fine-tuning | 42, 123, 2026 |

Repository layout:

```text
c2_struq_sft/
  seed_42/
  seed_123/
  seed_2026/

c3_secalign_dpo/
  seed_42/
  seed_123/
  seed_2026/

c4_ih_sft/
  seed_42/
  seed_123/
  seed_2026/
```

## Loading an adapter

Example:

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "{BASE_MODEL_ID}"
ADAPTER_REPO_ID = "{HF_REPO_ID}"
ADAPTER_SUBFOLDER = "c2_struq_sft/seed_42"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO_ID,
    subfolder=ADAPTER_SUBFOLDER,
)
```

## Experimental context

These adapters were produced for an academic experiment comparing preventive defenses against prompt injection in classification-style tasks.

The evaluated scenarios were:

```text
C0 — base model, no defense
C1 — StruQ format-only, no training
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Only C2, C3, and C4 have adapters.

## Limitations

- These adapters are experimental research artifacts.
- They were trained and evaluated on classification-style tasks.
- The adapters do not include the base model weights.
- The base model may require separate access approval.
- The training data and evaluation setup are specific to prompt-injection defense experiments.
- These adapters should not be interpreted as a general-purpose safety solution.

## Reproducibility

The experiment used three training seeds:

```text
42, 123, 2026
```

Each trained scenario has one adapter per seed.
"""

readme_path.write_text(model_card, encoding="utf-8")

print("README/model card gerado em:", readme_path)
print(readme_path.read_text(encoding="utf-8")[:2000])

README/model card gerado em: /workspace/pi-defense-exp/exports/huggingface_upload/README.md
---
base_model: meta-llama/Llama-3.1-8B-Instruct
library_name: peft
tags:
- peft
- lora
- qlora
- prompt-injection
- safety
- instruction-following
- experimental
---

# Prompt-Injection Defense Adapters

This repository contains LoRA/QLoRA adapters trained for an experimental evaluation of prompt-injection defenses.

The base model is:

```text
meta-llama/Llama-3.1-8B-Instruct
```

The base model is not included in this repository. Users must have access to the base model in order to load these adapters.

## Available adapters

The repository contains adapters for three trained scenarios:

| Scenario | Method | Seeds |
|---|---|---|
| C2 — StruQ-like SFT | Supervised fine-tuning | 42, 123, 2026 |
| C3 — SecAlign-like DPO | Preference optimization | 42, 123, 2026 |
| C4 — Instruction-Hierarchy-like SFT | Supervised fine-tuning | 42, 123, 2026 |

Repository layout:

```text
c2_struq_sft/
  seed_4

## 8. Gerar manifesto local dos adaptadores

Além do README, o notebook gera um manifesto JSON com a lista de adaptadores, caminhos locais, caminhos esperados no repositório, tamanhos e arquivos encontrados.

Esse manifesto é útil para auditoria e também será enviado para o repositório Hugging Face como:

```text
experiment_adapters_manifest.json
```

In [10]:
adapter_manifest = {
    "created_at_utc": utc_now(),
    "base_model_id": BASE_MODEL_ID,
    "hf_repo_id": HF_REPO_ID,
    "hf_repo_private": HF_REPO_PRIVATE,
    "dry_run": DRY_RUN,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "adapter_count": len(adapter_records),
    "total_adapter_size_mb": total_adapter_size_mb,
    "scenario_adapter_plan": {
        scenario_id: {
            **{
                key: str(value) if isinstance(value, Path) else value
                for key, value in info.items()
            }
        }
        for scenario_id, info in SCENARIO_ADAPTER_PLAN.items()
    },
    "upload_ignore_patterns": UPLOAD_IGNORE_PATTERNS,
    "adapters": [
        {
            **record,
            "local_path": str(record["local_path"]),
        }
        for record in adapter_records
    ],
}

adapter_manifest_path = EXPORT_DIR / "experiment_adapters_manifest.json"
write_json(adapter_manifest_path, adapter_manifest)

print("Manifesto dos adaptadores criado em:", adapter_manifest_path)

Manifesto dos adaptadores criado em: /workspace/pi-defense-exp/exports/huggingface_upload/experiment_adapters_manifest.json


## 9. Criar ou validar o repositório no Hugging Face

Esta etapa cria o repositório de destino, caso ele ainda não exista.

Como proteção, o notebook não executa nada enquanto `DRY_RUN=True`. Nesse modo, ele apenas mostra o que seria feito.

Antes de executar upload real, verifique:

```text
- o HF_REPO_ID está correto;
- o repositório deve ser privado ou público;
- o token autenticado tem permissão de escrita;
- os adaptadores foram validados;
- o README foi revisado.
```

In [11]:
if "CHANGE_ME" in HF_REPO_ID and not DRY_RUN:
    raise ValueError(
        "HF_REPO_ID ainda contém CHANGE_ME. "
        "Defina um repositório real antes de executar upload."
    )

repo_creation_record = {
    "repo_id": HF_REPO_ID,
    "repo_type": "model",
    "private": HF_REPO_PRIVATE,
    "dry_run": DRY_RUN,
    "status": None,
}

if DRY_RUN:
    print("[DRY RUN] Repositório não será criado.")
    print("[DRY RUN] Repo alvo:", HF_REPO_ID)
    repo_creation_record["status"] = "dry_run_skipped"
else:
    started_at = utc_now()
    start_time = time.time()

    try:
        api.create_repo(
            repo_id=HF_REPO_ID,
            repo_type="model",
            private=HF_REPO_PRIVATE,
            exist_ok=True,
        )

        repo_creation_record.update({
            "status": "created_or_existing",
            "started_at_utc": started_at,
            "finished_at_utc": utc_now(),
            "elapsed_seconds": time.time() - start_time,
        })

        print("Repositório criado ou já existente:", HF_REPO_ID)

    except Exception as error:
        repo_creation_record.update({
            "status": "failed",
            "error": repr(error),
            "traceback": traceback.format_exc(),
        })

        log_event("repo_creation_failed", repo_creation_record)
        raise

log_event("repo_creation_checked", repo_creation_record)
repo_creation_record

Repositório criado ou já existente: leinha/pi-defense-adapters


{'repo_id': 'leinha/pi-defense-adapters',
 'repo_type': 'model',
 'private': True,
 'dry_run': False,
 'status': 'created_or_existing',
 'started_at_utc': '2026-07-09T02:22:25.847089+00:00',
 'finished_at_utc': '2026-07-09T02:22:25.936382+00:00',
 'elapsed_seconds': 0.0892939567565918}

## 10. Enviar README e manifesto

Esta etapa envia os arquivos leves de documentação para o repositório:

```text
README.md
experiment_adapters_manifest.json
```

Esses arquivos são pequenos e ajudam qualquer pessoa que acesse o repositório a entender o que foi publicado.

Em `DRY_RUN=True`, a célula apenas mostra o que seria enviado.

In [12]:
metadata_upload_records = []

metadata_files_to_upload = []

if UPLOAD_README:
    metadata_files_to_upload.append({
        "local_path": readme_path,
        "path_in_repo": "README.md",
        "description": "Repository model card",
    })

if UPLOAD_ADAPTER_MANIFEST:
    metadata_files_to_upload.append({
        "local_path": adapter_manifest_path,
        "path_in_repo": "experiment_adapters_manifest.json",
        "description": "Adapter manifest",
    })

for item in metadata_files_to_upload:
    local_path = item["local_path"]
    path_in_repo = item["path_in_repo"]

    record = {
        "kind": "metadata_file",
        "description": item["description"],
        "local_path": str(local_path),
        "path_in_repo": path_in_repo,
        "dry_run": DRY_RUN,
        "status": None,
    }

    if DRY_RUN:
        print(f"[DRY RUN] Enviaria {local_path} para {HF_REPO_ID}/{path_in_repo}")
        record["status"] = "dry_run_skipped"
    else:
        started_at = utc_now()
        start_time = time.time()

        try:
            api.upload_file(
                repo_id=HF_REPO_ID,
                repo_type="model",
                path_or_fileobj=str(local_path),
                path_in_repo=path_in_repo,
                commit_message=f"Upload {path_in_repo}",
            )

            record.update({
                "status": "uploaded",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now(),
                "elapsed_seconds": time.time() - start_time,
            })

            print("Arquivo enviado:", path_in_repo)

        except Exception as error:
            record.update({
                "status": "failed",
                "error": repr(error),
                "traceback": traceback.format_exc(),
            })

            log_event("metadata_upload_failed", record)
            raise

    metadata_upload_records.append(record)
    log_event("metadata_upload_checked", record)

metadata_upload_df = pd.DataFrame(metadata_upload_records)
display(metadata_upload_df)

No files have been modified since last commit. Skipping to prevent empty commit.


Arquivo enviado: README.md
Arquivo enviado: experiment_adapters_manifest.json


,kind,description,local_path,path_in_repo,dry_run,status,started_at_utc,finished_at_utc,elapsed_seconds
0,metadata_file,Repository model card,/workspace/pi-defense-exp/exports/huggingface_...,README.md,False,uploaded,2026-07-09T02:22:25.942531+00:00,2026-07-09T02:22:26.101021+00:00,0.158491
1,metadata_file,Adapter manifest,/workspace/pi-defense-exp/exports/huggingface_...,experiment_adapters_manifest.json,False,uploaded,2026-07-09T02:22:26.101166+00:00,2026-07-09T02:22:26.498874+00:00,0.397712


## 11. Upload dos adaptadores

Esta é a etapa principal do notebook.

Para cada combinação de cenário e seed, o notebook envia uma pasta local de adaptador para o caminho correspondente no repositório Hugging Face.

O upload usa uma lista de padrões ignorados para evitar enviar checkpoints intermediários e arquivos de treinamento desnecessários.

Por exemplo, a pasta local:

```text
/workspace/pi-defense-exp/adapters/struq/seed_42/
```

é enviada para:

```text
c2_struq_sft/seed_42/
```

no repositório do Hugging Face.

Em `DRY_RUN=True`, nenhum arquivo é enviado.

Nesta etapa, cada adaptador é enviado diretamente do seu diretório original. O notebook não cria staging local nem copia `adapter_model.safetensors` para `exports/`.

Isso é importante porque os adaptadores podem ocupar bastante espaço quando há múltiplos cenários e seeds. A pasta `exports/huggingface_upload/` continua sendo usada apenas para metadados leves, como README, manifesto e logs do upload.


In [13]:
adapter_upload_records = []

if UPLOAD_ADAPTERS:
    for record in adapter_records:
        local_path = Path(record["local_path"])
        path_in_repo = record["path_in_repo"]

        upload_record = {
            "kind": "adapter_folder",
            "scenario_id": record["scenario_id"],
            "scenario_label": record["scenario_label"],
            "method": record["method"],
            "seed": record["seed"],
            "local_path": str(local_path),
            "path_in_repo": path_in_repo,
            "file_count": record["file_count"],
            "size_mb": record["size_mb"],
            "dry_run": DRY_RUN,
            "status": None,
        }

        print()
        print("=" * 80)
        print(f"Adapter: {record['scenario_id']} | seed={record['seed']}")
        print("Local:", local_path)
        print("Repo path:", path_in_repo)
        print(f"Size: {record['size_mb']:.2f} MB")
        print("=" * 80)

        if DRY_RUN:
            print(f"[DRY RUN] Enviaria pasta {local_path} para {HF_REPO_ID}/{path_in_repo}")
            upload_record["status"] = "dry_run_skipped"
        else:
            started_at = utc_now()
            start_time = time.time()

            try:
                api.upload_folder(
                    repo_id=HF_REPO_ID,
                    repo_type="model",
                    folder_path=str(local_path),
                    path_in_repo=path_in_repo,
                    ignore_patterns=UPLOAD_IGNORE_PATTERNS,
                    commit_message=(
                        f"Upload {record['scenario_id']} seed {record['seed']} adapter"
                    ),
                )

                upload_record.update({
                    "status": "uploaded",
                    "started_at_utc": started_at,
                    "finished_at_utc": utc_now(),
                    "elapsed_seconds": time.time() - start_time,
                })

                print("Upload concluído:", path_in_repo)

            except Exception as error:
                upload_record.update({
                    "status": "failed",
                    "error": repr(error),
                    "traceback": traceback.format_exc(),
                })

                error_path = LOG_DIR / f"upload_error_{record['scenario_id']}_seed_{record['seed']}.txt"
                error_path.write_text(traceback.format_exc(), encoding="utf-8")
                upload_record["error_path"] = str(error_path)

                log_event("adapter_upload_failed", upload_record)
                print("Falha no upload. Erro registrado em:", error_path)
                raise

        adapter_upload_records.append(upload_record)
        log_event("adapter_upload_checked", upload_record)

else:
    print("UPLOAD_ADAPTERS=False; adaptadores não serão enviados.")

adapter_upload_df = pd.DataFrame(adapter_upload_records)
display(adapter_upload_df)


Adapter: c2_struq_sft | seed=42
Local: /workspace/pi-defense-exp/adapters/struq/seed_42
Repo path: c2_struq_sft/seed_42
Size: 610.24 MB


No files have been modified since last commit. Skipping to prevent empty commit.


Upload concluído: c2_struq_sft/seed_42

Adapter: c2_struq_sft | seed=123
Local: /workspace/pi-defense-exp/adapters/struq/seed_123
Repo path: c2_struq_sft/seed_123
Size: 610.24 MB


No files have been modified since last commit. Skipping to prevent empty commit.


Upload concluído: c2_struq_sft/seed_123

Adapter: c2_struq_sft | seed=2026
Local: /workspace/pi-defense-exp/adapters/struq/seed_2026
Repo path: c2_struq_sft/seed_2026
Size: 610.24 MB


...

No files have been modified since last commit. Skipping to prevent empty commit.


Upload concluído: c4_ih_sft/seed_123

Adapter: c4_ih_sft | seed=2026
Local: /workspace/pi-defense-exp/adapters/ih/seed_2026
Repo path: c4_ih_sft/seed_2026
Size: 610.24 MB


No files have been modified since last commit. Skipping to prevent empty commit.


Upload concluído: c4_ih_sft/seed_2026


,kind,scenario_id,scenario_label,method,seed,local_path,path_in_repo,file_count,size_mb,dry_run,status,started_at_utc,finished_at_utc,elapsed_seconds
0,adapter_folder,c2_struq_sft,C2 — StruQ-like SFT,sft,42,/workspace/pi-defense-exp/adapters/struq/seed_42,c2_struq_sft/seed_42,29,610.235897,False,uploaded,2026-07-09T02:22:26.508823+00:00,2026-07-09T02:22:28.641892+00:00,2.133069
1,adapter_folder,c2_struq_sft,C2 — StruQ-like SFT,sft,123,/workspace/pi-defense-exp/adapters/struq/seed_123,c2_struq_sft/seed_123,29,610.236034,False,uploaded,2026-07-09T02:22:28.642140+00:00,2026-07-09T02:22:35.252329+00:00,6.610194
2,adapter_folder,c2_struq_sft,C2 — StruQ-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/struq/seed_...,c2_struq_sft/seed_2026,29,610.235914,False,uploaded,2026-07-09T02:22:35.252621+00:00,2026-07-09T02:22:42.261770+00:00,7.009156
3,adapter_folder,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,42,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_42,29,610.238959,False,uploaded,2026-07-09T02:22:42.262067+00:00,2026-07-09T02:22:43.052805+00:00,0.790743
4,adapter_folder,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,123,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_123,29,610.238979,False,uploaded,2026-07-09T02:22:43.053052+00:00,2026-07-09T02:22:47.748436+00:00,4.695389
5,adapter_folder,c3_secalign_dpo,C3 — SecAlign-like DPO,dpo,2026,/workspace/pi-defense-exp/adapters/secalign/se...,c3_secalign_dpo/seed_2026,29,610.238911,False,uploaded,2026-07-09T02:22:47.748870+00:00,2026-07-09T02:23:02.721412+00:00,14.972546
6,adapter_folder,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,42,/workspace/pi-defense-exp/adapters/ih/seed_42,c4_ih_sft/seed_42,29,610.235907,False,uploaded,2026-07-09T02:23:02.721684+00:00,2026-07-09T02:23:12.598275+00:00,9.876596
7,adapter_folder,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,123,/workspace/pi-defense-exp/adapters/ih/seed_123,c4_ih_sft/seed_123,29,610.236018,False,uploaded,2026-07-09T02:23:12.598514+00:00,2026-07-09T02:23:13.651356+00:00,1.052852
8,adapter_folder,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/ih/seed_2026,c4_ih_sft/seed_2026,29,610.235898,False,uploaded,2026-07-09T02:23:13.651831+00:00,2026-07-09T02:23:18.699998+00:00,5.048171


## 12. Verificação remota

Depois do upload real, esta etapa pode listar os arquivos no repositório remoto.

Essa verificação ajuda a confirmar que o Hub recebeu os arquivos esperados. Em `DRY_RUN=True`, a verificação remota é pulada.

In [14]:
remote_files = []

if VERIFY_REMOTE_FILES and not DRY_RUN:
    try:
        remote_files = api.list_repo_files(
            repo_id=HF_REPO_ID,
            repo_type="model",
        )

        print(f"Arquivos remotos encontrados: {len(remote_files)}")

        remote_files_path = EXPORT_DIR / "remote_files.json"
        write_json(remote_files_path, {
            "repo_id": HF_REPO_ID,
            "checked_at_utc": utc_now(),
            "remote_file_count": len(remote_files),
            "remote_files": remote_files,
        })

        display(pd.DataFrame({"path": remote_files}).head(50))

        log_event("remote_files_checked", {
            "repo_id": HF_REPO_ID,
            "remote_file_count": len(remote_files),
            "remote_files_path": str(remote_files_path),
        })

    except Exception as error:
        log_event("remote_files_check_failed", {
            "repo_id": HF_REPO_ID,
            "error": repr(error),
        })
        raise

else:
    if DRY_RUN:
        print("[DRY RUN] Verificação remota pulada.")
    else:
        print("VERIFY_REMOTE_FILES=False; verificação remota pulada.")

Arquivos remotos encontrados: 57


,path
0,.gitattributes
1,README.md
2,c2_struq_sft/seed_123/README.md
3,c2_struq_sft/seed_123/adapter_config.json
4,c2_struq_sft/seed_123/adapter_model.safetensors
5,c2_struq_sft/seed_123/chat_template.jinja
6,c2_struq_sft/seed_123/tokenizer.json
7,c2_struq_sft/seed_123/tokenizer_config.json
8,c2_struq_sft/seed_2026/README.md
9,c2_struq_sft/seed_2026/adapter_config.json


## 13. Como carregar os adaptadores depois do upload

Depois que o upload real for concluído, os adaptadores podem ser carregados com `PeftModel.from_pretrained`.

Exemplo para o adaptador C2 com seed 42:

```python
from peft import PeftModel

ADAPTER_REPO_ID = "usuario/pi-defense-adapters"
ADAPTER_SUBFOLDER = "c2_struq_sft/seed_42"

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO_ID,
    subfolder=ADAPTER_SUBFOLDER,
)
```

O mesmo padrão vale para os demais cenários:

```text
c2_struq_sft/seed_123
c2_struq_sft/seed_2026

c3_secalign_dpo/seed_42
c3_secalign_dpo/seed_123
c3_secalign_dpo/seed_2026

c4_ih_sft/seed_42
c4_ih_sft/seed_123
c4_ih_sft/seed_2026
```

O modelo base precisa ser carregado separadamente antes da aplicação do adaptador.

In [15]:
loading_examples = {
    "repo_id": HF_REPO_ID,
    "base_model_id": BASE_MODEL_ID,
    "examples": [
        {
            "scenario_id": record["scenario_id"],
            "seed": record["seed"],
            "adapter_subfolder": record["path_in_repo"],
            "code": (
                "model = PeftModel.from_pretrained("
                "base_model, "
                f'"{HF_REPO_ID}", '
                f'subfolder="{record["path_in_repo"]}"'
                ")"
            ),
        }
        for record in adapter_records
    ],
}

loading_examples_path = EXPORT_DIR / "adapter_loading_examples.json"
write_json(loading_examples_path, loading_examples)

print("Exemplos de carregamento salvos em:", loading_examples_path)
display(pd.DataFrame(loading_examples["examples"]))

Exemplos de carregamento salvos em: /workspace/pi-defense-exp/exports/huggingface_upload/adapter_loading_examples.json


,scenario_id,seed,adapter_subfolder,code
0,c2_struq_sft,42,c2_struq_sft/seed_42,"model = PeftModel.from_pretrained(base_model, ..."
1,c2_struq_sft,123,c2_struq_sft/seed_123,"model = PeftModel.from_pretrained(base_model, ..."
2,c2_struq_sft,2026,c2_struq_sft/seed_2026,"model = PeftModel.from_pretrained(base_model, ..."
3,c3_secalign_dpo,42,c3_secalign_dpo/seed_42,"model = PeftModel.from_pretrained(base_model, ..."
4,c3_secalign_dpo,123,c3_secalign_dpo/seed_123,"model = PeftModel.from_pretrained(base_model, ..."
5,c3_secalign_dpo,2026,c3_secalign_dpo/seed_2026,"model = PeftModel.from_pretrained(base_model, ..."
6,c4_ih_sft,42,c4_ih_sft/seed_42,"model = PeftModel.from_pretrained(base_model, ..."
7,c4_ih_sft,123,c4_ih_sft/seed_123,"model = PeftModel.from_pretrained(base_model, ..."
8,c4_ih_sft,2026,c4_ih_sft/seed_2026,"model = PeftModel.from_pretrained(base_model, ..."


## 14. Manifesto final do upload

Esta etapa gera o manifesto final do notebook 11.

O manifesto registra:

```text
- repositório de destino;
- usuário autenticado;
- modelo base;
- modo dry-run;
- adaptadores planejados;
- arquivos enviados;
- arquivos de metadados;
- verificação remota;
- logs gerados.
```

Esse manifesto permite auditar exatamente o que foi enviado ou o que teria sido enviado em modo dry-run.

In [16]:
manifest = {
    "notebook": "11_upload_adapters_to_huggingface",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "base_model_id": BASE_MODEL_ID,
    "hf_repo_id": HF_REPO_ID,
    "hf_repo_private": HF_REPO_PRIVATE,
    "hf_user": HF_USERNAME,
    "dry_run": DRY_RUN,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "upload_ignore_patterns": UPLOAD_IGNORE_PATTERNS,
    "repo_creation": repo_creation_record,
    "adapter_manifest_path": str(adapter_manifest_path),
    "readme_path": str(readme_path),
    "loading_examples_path": str(loading_examples_path),
    "metadata_upload_records": metadata_upload_records,
    "adapter_upload_records": adapter_upload_records,
    "remote_file_count": len(remote_files),
    "remote_files_path": str(EXPORT_DIR / "remote_files.json") if remote_files else None,
    "events_log_path": str(EVENTS_LOG_PATH),
    "no_intermediate_adapter_copy": NO_INTERMEDIATE_ADAPTER_COPY,
    "upload_metadata_dir": str(UPLOAD_METADATA_DIR),
}

manifest_json_path = MANIFEST_DIR / "11_upload_adapters_to_huggingface_manifest.json"
write_json(manifest_json_path, manifest)

print("Manifesto JSON criado em:", manifest_json_path)

Manifesto JSON criado em: /workspace/pi-defense-exp/manifests/huggingface_upload/11_upload_adapters_to_huggingface_manifest.json


## 15. Manifesto Markdown

Além do manifesto JSON, o notebook gera uma versão em Markdown para leitura manual.

Essa versão resume o estado final do upload e deve ser suficiente para identificar rapidamente:

```text
- se foi dry-run ou upload real;
- qual repositório foi usado;
- quantos adaptadores foram considerados;
- quantos uploads foram concluídos;
- quais arquivos auxiliares foram gerados.
```

In [17]:
def markdown_table_from_records(records: list[dict], columns: list[str]) -> str:
    if not records:
        return "_Nenhum registro._"

    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]

    for record in records:
        values = []

        for column in columns:
            value = record.get(column, "")

            if isinstance(value, float):
                value = f"{value:.6f}"
            else:
                value = str(value)

            value = value.replace("|", "\\|")
            values.append(value)

        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)


adapter_table_md = markdown_table_from_records(
    [
        {
            "scenario_id": record["scenario_id"],
            "seed": record["seed"],
            "path_in_repo": record["path_in_repo"],
            "status": next(
                (
                    upload_record.get("status")
                    for upload_record in adapter_upload_records
                    if upload_record.get("scenario_id") == record["scenario_id"]
                    and upload_record.get("seed") == record["seed"]
                ),
                "not_uploaded",
            ),
            "size_mb": record["size_mb"],
        }
        for record in adapter_records
    ],
    ["scenario_id", "seed", "path_in_repo", "status", "size_mb"],
)

metadata_table_md = markdown_table_from_records(
    metadata_upload_records,
    ["description", "path_in_repo", "status"],
)

manifest_md = f"""# Manifesto — Upload dos adaptadores para Hugging Face

## Identificação

- Notebook: `11_upload_adapters_to_huggingface`
- Gerado em UTC: `{manifest["created_at_utc"]}`
- Repositório alvo: `{HF_REPO_ID}`
- Repositório privado: `{HF_REPO_PRIVATE}`
- Usuário autenticado: `{HF_USERNAME}`
- Dry-run: `{DRY_RUN}`

## Modelo base

```text
{BASE_MODEL_ID}
```

O modelo base não foi enviado por este notebook. Apenas os adaptadores LoRA/QLoRA são considerados para upload.

## Arquivos auxiliares

| Arquivo | Caminho |
|---|---|
| README/model card | `{readme_path}` |
| Manifesto dos adaptadores | `{adapter_manifest_path}` |
| Exemplos de carregamento | `{loading_examples_path}` |
| Log de eventos | `{EVENTS_LOG_PATH}` |

## Metadados enviados

{metadata_table_md}

## Adaptadores

{adapter_table_md}

## Verificação remota

- Verificação remota executada: `{VERIFY_REMOTE_FILES and not DRY_RUN}`
- Total de arquivos remotos listados: `{len(remote_files)}`

## Observações

- Em modo `DRY_RUN=True`, nenhum repositório é criado e nenhum arquivo é enviado.
- Checkpoints intermediários são ignorados pelos padrões definidos em `UPLOAD_IGNORE_PATTERNS`.
- O repositório deve ser revisado antes de ser tornado público.
- Tokens Hugging Face não são registrados nos manifestos.
"""

manifest_md_path = MANIFEST_DIR / "11_upload_adapters_to_huggingface_manifest.md"
manifest_md_path.write_text(manifest_md, encoding="utf-8")

print("Manifesto Markdown criado em:", manifest_md_path)

Manifesto Markdown criado em: /workspace/pi-defense-exp/manifests/huggingface_upload/11_upload_adapters_to_huggingface_manifest.md


## 16. Próximos passos

Depois de validar este notebook em `DRY_RUN=True`, os próximos passos são:

```text
1. Definir HF_REPO_ID com o namespace correto.
2. Revisar README.md gerado.
3. Confirmar que o repositório deve ser privado ou público.
4. Alterar DRY_RUN para False.
5. Executar novamente as células de criação e upload.
6. Verificar os arquivos remotos no Hugging Face.
7. Testar o carregamento de pelo menos um adaptador a partir do Hub.
```

A validação mais importante depois do upload é tentar carregar um adaptador do Hub em uma sessão limpa.

O teste mínimo é:

```text
- carregar o modelo base;
- aplicar um adaptador com PeftModel.from_pretrained;
- gerar uma resposta curta;
- confirmar que não há erro de caminho, permissão ou configuração.
```

Se o repositório for público, revise cuidadosamente o README, os nomes dos cenários e os arquivos enviados antes de divulgar.